#### Just starter -- actual project code is after this code cell

In [2]:
import os
from huggingface_hub import InferenceClient
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv()

# Initialize the client with the inference provider
client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

# ── Step 1:
model_small = "sentence-transformers/all-MiniLM-L6-v2"
model_large = "sentence-transformers/all-mpnet-base-v2"


# ── Step 2: A small test set with known relationships ─────────────
sentences = [
"The university requires a CNIC copy for admission.",   # 0
"Students must submit a copy of their national ID card.",  # 1 -- paraphrase of 0
"The cafeteria is open from 8am to 6pm.", # 2 -- unrelated
"Admission documents include the CNIC and academic transcripts.", # 3 -- related to 0
]


# ── Step 3: Encode with both models (normalised for fair comparison) ──
emb_small = client.feature_extraction(text=sentences,model=model_small,normalize=True)
emb_large = client.feature_extraction(text=sentences,model=model_large,normalize=True)


print("Small model dims:", emb_small.shape)
print("Large model dims:", emb_large.shape)


# ── Step 4: Compare similarity for the same pairs across both models ──
def report(emb,name):
    sims = cosine_similarity(emb)
    print(f"----{name}----")
    print(f"Paraphrase pair (0 vs 1) : {sims[0][1]:.4f}")
    print(f"Unrelated pair (0 vs 2) : {sims[0][2]:.4f}")
    print(f"Related pair (0 vs 3) : {sims[0][3]:.4f}")


report(emb_small, "all-MiniLM-L6-v2")
report(emb_large, "all-mpnet-base-v2")


/home/madiha/My Notes/2026/Code/RAG (basic to advance)/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Small model dims: (4, 384)
Large model dims: (4, 768)
----all-MiniLM-L6-v2----
Paraphrase pair (0 vs 1) : 0.5093
Unrelated pair (0 vs 2) : 0.0627
Related pair (0 vs 3) : 0.7684
----all-mpnet-base-v2----
Paraphrase pair (0 vs 1) : 0.6209
Unrelated pair (0 vs 2) : 0.1147
Related pair (0 vs 3) : 0.8056


### Capstone start

In [2]:
import os
from huggingface_hub import InferenceClient
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv()

# Initialize the client with the inference provider
client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

model_large = "sentence-transformers/all-mpnet-base-v2"


my_chunks = [
"students must complete Semester registration before the start of classes.However, students will be allowed to add and drop courses as per the academic calendar with thepermission of the concerned HoD..",  

"One credit hour means dir ect teaching a theory course for a mi ni mum of 15 academic hours per semester. In case of practical one credit hour means direct instructions as well as performance of the experiments in laboratory environment for a minimum duration of 45 academic hours. An academic hour at the University is of 50 minutes duration.", 

"Before the submission of PhD thesis, a scholar must have contributed as a sole author or as a co-author to at least one article, related to his research work presented in the dissertation, published or accepted for publication (with acceptance letter from the Editor / Sub-editor of the journal) in international ISI indexed journal and or acceptable to the HEC.",

"The academic standing of a student is considered excellent if he/ achieves a Semester-GPA >=3.67 and CGPA >=3.33 Good The academic performance of a student in a semester is considered good if his/ Semester GPA<3.67 & >=3.00A is less than",

"At the end of period of expulsion, the student may resume his studies. However, the resumption could only be made in the start of a regular semester. In all such cases, the concerned Head of Teaching Department shall have the authority to notify the students expelled from the University with copies to all concerned offices/authorities."

]
my_embeddings = client.feature_extraction(text=my_chunks,model=model_large,normalize=True)
print("Capstone embeddings shape:", my_embeddings.shape)

Capstone embeddings shape: (5, 768)
